In [6]:
import sys
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Folder:", Path.cwd().name)

for name in ["numpy", "pandas", "sklearn"]:
    try:
        __import__(name)
        print(name, "- ok")
    except ImportError:
        print(name, "- missing")

Python: 3.14.6
Folder: P01-workbench
numpy - ok
pandas - ok
sklearn - ok


In [7]:
import csv
from pathlib import Path
import numpy as np

SEED = 42
N_ROWS = 600
DATA = Path("..") / "data" / "delivery_times.csv"

def make_delivery_csv(path=DATA):
    rng = np.random.default_rng(SEED)
    distance_km = np.round(rng.uniform(0.5, 12.0, N_ROWS), 2)
    prep_time_min = np.round(rng.uniform(5, 30, N_ROWS), 0)
    traffic_level = rng.integers(1, 4, N_ROWS)
    rain = rng.binomial(1, 0.25, N_ROWS)
    delivery_min = np.round(
        6.0 + 3.1 * distance_km + 0.65 * prep_time_min
        + 4.2 * traffic_level + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS), 1)

    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["distance_km", "prep_time_min", "traffic_level", "rain", "delivery_min"])
        for i in range(N_ROWS):
            w.writerow([distance_km[i], int(prep_time_min[i]),
                        int(traffic_level[i]), int(rain[i]), delivery_min[i]])
    return path

if not DATA.exists():
    make_delivery_csv()
print("dataset ready:", DATA)

dataset ready: ..\data\delivery_times.csv


In [8]:
print("Python version:", sys.version.split()[0])
print("Python program:", sys.executable)
print("Inside .venv :", ".venv" in sys.executable)

Python version: 3.14.6
Python program: C:\ProgramData\anaconda3\python.exe
Inside .venv : False


In [9]:
from importlib.metadata import version

LIBRARIES = ["numpy", "pandas", "scikit-learn", "matplotlib"]
for name in LIBRARIES:
    print(name, version(name))

numpy 2.4.6
pandas 3.0.3
scikit-learn 1.9.0
matplotlib 3.11.0


In [10]:
WORK = Path("work")
WORK.mkdir(exist_ok=True)

lines = [f"{name}=={version(name)}" for name in LIBRARIES]
(WORK / "requirements.txt").write_text("\n".join(lines) + "\n")
print((WORK / "requirements.txt").read_text())

numpy==2.4.6
pandas==3.0.3
scikit-learn==1.9.0
matplotlib==3.11.0



In [11]:
careless = np.random.default_rng()
print(np.round(careless.uniform(0, 10, 3), 2))

[8.05 6.79 0.79]


In [12]:
first = np.random.default_rng(42).uniform(0, 10, 3)
second = np.random.default_rng(42).uniform(0, 10, 3)
print(np.round(first, 2))
print(np.round(second, 2))
print("identical:", np.array_equal(first, second))

[7.74 4.39 8.59]
[7.74 4.39 8.59]
identical: True


In [13]:
import hashlib

def sha256_of(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

make_delivery_csv(WORK / "run_a.csv")
make_delivery_csv(WORK / "run_b.csv")
print("identical files:", sha256_of(WORK / "run_a.csv") == sha256_of(WORK / "run_b.csv"))

identical files: True


In [14]:
import pandas as pd

orders = pd.read_csv(DATA)
print(orders.shape)
print(orders.head())
print(orders.describe().round(1))

(600, 5)
   distance_km  prep_time_min  traffic_level  rain  delivery_min
0         9.40             17              1     0          51.3
1         5.55             24              2     1          54.2
2        10.37             28              3     0          67.7
3         8.52             23              2     0          51.2
4         1.58             29              2     1          42.1
       distance_km  prep_time_min  traffic_level   rain  delivery_min
count        600.0          600.0          600.0  600.0         600.0
mean           6.2           17.6            2.0    0.3          46.6
std            3.3            7.4            0.8    0.4          12.1
min            0.6            5.0            1.0    0.0          18.5
25%            3.2           11.0            1.0    0.0          37.8
50%            6.2           17.0            2.0    0.0          46.5
75%            9.1           24.0            3.0    1.0          55.7
max           12.0           30.0        

In [15]:
import json

run_info = {
    "python": sys.version.split()[0],
    "seed": SEED,
    "rows": len(orders),
    "data_sha256": sha256_of(DATA),
    "libraries": {n: version(n) for n in LIBRARIES},
}
(WORK / "run_info.json").write_text(json.dumps(run_info, indent=2))
print(json.dumps(run_info, indent=2))

{
  "python": "3.14.6",
  "seed": 42,
  "rows": 600,
  "data_sha256": "9e9f7a46c817d5bb81c3e458f66617d17a116a3218c6bb5f555c05dd831f59a1",
  "libraries": {
    "numpy": "2.4.6",
    "pandas": "3.0.3",
    "scikit-learn": "1.9.0",
    "matplotlib": "3.11.0"
  }
}


In [16]:
import subprocess

def git(*args):
    result = subprocess.run(["git", *args], cwd=WORK, capture_output=True, text=True)
    print(result.stdout + result.stderr)

git("init", "-q")
git("config", "user.name", "SCSE3040 Student")
git("config", "user.email", "student@bennett.edu.in")
git("add", "requirements.txt", "run_info.json")
git("commit", "-q", "-m", "P01: pinned requirements and run record")
git("log", "--oneline")






67550ab P01: pinned requirements and run record



In [17]:
seed7 = np.random.default_rng(7)
distance_km_t1 = np.round(seed7.uniform(0.5, 12.0, N_ROWS), 2)
print(distance_km_t1[:3])

[ 7.69 10.82  9.42]


In [18]:
LIBRARIES_T2 = ["numpy", "pandas", "scikit-learn"]
lines_t2 = [f"{name}=={version(name)}" for name in LIBRARIES_T2]
(WORK / "my_requirements.txt").write_text("\n".join(lines_t2) + "\n")
print((WORK / "my_requirements.txt").read_text())

numpy==2.4.6
pandas==3.0.3
scikit-learn==1.9.0



In [19]:
def fingerprint(path):
    rows = len(pd.read_csv(path))
    return {
        "rows": rows,
        "sha256": sha256_of(path),
        "seed": SEED,
    }

print(fingerprint(DATA))

{'rows': 600, 'sha256': '9e9f7a46c817d5bb81c3e458f66617d17a116a3218c6bb5f555c05dd831f59a1', 'seed': 42}
